In [7]:
import argparse
import json
import logging
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay,
    ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# ---------- fixed config ----------
CSV_PATH = "/content/german_credit_data.csv"
PLOTS_DIR = "plots"

@dataclass
class TrainConfig:
    model_name: str
    tune: bool
    test_size: float  # interpreted as the fraction for TEST and also for VALIDATION (so train = 1 - 2*test_size)
    seed: int
    make_plots: bool


def load_data() -> pd.DataFrame:
    df = pd.read_csv(CSV_PATH)
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]
    return df


def split_features_target(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    risk_col = "Risk" if "Risk" in df.columns else [c for c in df.columns if c.lower() == "risk"][0]
    y = df[risk_col]
    if y.dtype == object:
        y = y.astype("category").cat.codes
    X = df.drop(columns=[risk_col])
    return X, y


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    return ColumnTransformer([
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe, num_cols),
    ])


def get_models(seed: int) -> Dict[str, Pipeline]:
    return {
        "lr": Pipeline([("clf", LogisticRegression(max_iter=2000, random_state=seed))]),
        "knn": Pipeline([("clf", KNeighborsClassifier())]),
        "dt": Pipeline([("clf", DecisionTreeClassifier(random_state=seed))]),
        "rf": Pipeline([("clf", RandomForestClassifier(random_state=seed))]),
    }


def get_param_distributions(model_name: str) -> Dict:
    if model_name == "lr":
        return {"clf__C": np.logspace(-3, 3, 20), "clf__penalty": ["l2"], "clf__solver": ["lbfgs"]}
    if model_name == "knn":
        return {"clf__n_neighbors": np.arange(1, 51), "clf__weights": ["uniform", "distance"]}
    if model_name == "dt":
        return {"clf__max_depth": np.arange(2, 31), "clf__criterion": ["gini", "entropy"]}
    if model_name == "rf":
        return {
            "clf__n_estimators": np.arange(50, 501, 25),
            "clf__max_depth": [None] + list(np.arange(3, 31)),
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf": [1, 2, 4],
            "clf__criterion": ["gini", "entropy"],
        }
    return {}


def evaluate_and_plot(model: Pipeline, X_test, y_test, make_plots: bool, label: str) -> Dict[str, float]:
    y_pred = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }

    # ROC-AUC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)
    metrics["roc_auc"] = roc_auc_score(y_test, y_score)

    if make_plots:
        import os
        os.makedirs(PLOTS_DIR, exist_ok=True)
        RocCurveDisplay.from_predictions(y_test, y_score)
        plt.title(f"ROC Curve — {label}")
        plt.savefig(f"{PLOTS_DIR}/roc_{label}.png", dpi=150)
        plt.close()

        cm = confusion_matrix(y_test, y_pred)
        ConfusionMatrixDisplay(confusion_matrix=cm).plot()
        plt.title(f"Confusion Matrix — {label}")
        plt.savefig(f"{PLOTS_DIR}/cm_{label}.png", dpi=150)
        plt.close()

    return metrics


def cross_validate(model: Pipeline, X, y, seed: int) -> float:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    return float(np.mean(scores))


def parse_args(argv: Optional[List[str]] = None) -> TrainConfig:
    p = argparse.ArgumentParser(description="Financial Risk — Training & Evaluation")
    p.add_argument("--model", dest="model_name", type=str, default="rf",
                   choices=["lr", "knn", "dt", "rf"])
    p.add_argument("--tune", action="store_true")
    # test_size used for BOTH validation and test (so train = 1 - 2*test_size)
    p.add_argument("--test-size", type=float, default=0.20)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--no-plots", action="store_true")
    args, _ = p.parse_known_args(argv)
    if not (0 < args.test_size < 0.5):
        raise ValueError("--test-size must be in (0, 0.5)")
    return TrainConfig(args.model_name, args.tune, args.test_size, args.seed, not args.no_plots)


def main(argv: Optional[List[str]] = None) -> int:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    cfg = parse_args(argv)

    df = load_data()
    X, y = split_features_target(df)
    preprocessor = build_preprocessor(X)

    models = get_models(cfg.seed)
    base_model = models[cfg.model_name].named_steps["clf"]
    pipe = Pipeline([("pre", preprocessor), ("clf", base_model)])

    # -------- Train / Val / Test Split (60/20/20) --------
    temp_size = 2 * cfg.test_size
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=temp_size, random_state=cfg.seed, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=cfg.seed, stratify=y_temp
    )

    logging.info(f"Data split sizes: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

    if cfg.tune:
        param_distributions = get_param_distributions(cfg.model_name)
        rs = RandomizedSearchCV(
            pipe, param_distributions, n_iter=40, scoring="roc_auc",
            cv=5, random_state=cfg.seed, n_jobs=-1, verbose=1
        )
        rs.fit(X_train, y_train)
        model = rs.best_estimator_
        cv_score = rs.best_score_
    else:
        model = pipe.fit(X_train, y_train)
        cv_score = cross_validate(model, X_train, y_train, cfg.seed)

    # Validation performance
    if hasattr(model, "predict_proba"):
        y_val_score = model.predict_proba(X_val)[:, 1]
    else:
        y_val_score = model.decision_function(X_val)
    val_auc = roc_auc_score(y_val, y_val_score)
    logging.info(f"Validation ROC-AUC: {val_auc:.3f}")

    # Final test evaluation
    metrics = evaluate_and_plot(model, X_test, y_test, cfg.make_plots, cfg.model_name)

    print(json.dumps({
        "model": cfg.model_name,
        "tuned": cfg.tune,
        "cv_roc_auc": cv_score,
        "val_roc_auc": float(val_auc),
        **metrics
    }, indent=2))
    return 0


if __name__ == "__main__":
    main([])


{
  "model": "rf",
  "tuned": false,
  "cv_roc_auc": 0.6278769841269842,
  "val_roc_auc": 0.6861309523809525,
  "accuracy": 0.66,
  "precision": 0.725,
  "recall": 0.8285714285714286,
  "f1": 0.7733333333333333,
  "roc_auc": 0.5797023809523809
}
